# 🌍 CO₂ Emissions & Economic Growth — EDA & Hypothesis Testing

**Author:** `[YOUR NAME]`  
**Date:** `[DATE]`  
**Branch:** `[YOUR GITHUB BRANCH]`

---
## Business Case

A group of environmental activits is following closely how international politics and agreements effectively support the worring environmental situation of climate change. This time the research group is interested to evaluate the time window coinciding with post-Paris Agreement policy momentum, and find out 
**Which regions show evidence that renewable penetration explains CO₂ intensity decline — and where is the gap largest?.**

### Hypothesis 1

> **CO₂ intensity of GDP (co2/GDP) has declined across regions from 2014–2021?.**

bla bal bal

### Hypothesis 2
Following the hypothesis 1, then it is interesting to see the mechanism used for the CO2 intensity decline and see why it happens, here we want to see:
> **Which regions show evidence that renewable penetration explains CO₂ intensity decline — and where is the gap largest?.**
---
> **Scenario A: co2 / GDP       → fossil+industrial only → optimistic view.**

> **Scenario B: co2_luc / GDP   → full footprint         → honest view.**

Scenario can tell us: if decoupling holds in A but weakens in B, means --> high-income countries shifted emissions, not eliminated them.

---

## Notebook Purpose

This notebook performs:
1. **Exploratory Data Analysis (EDA)** — structure, distributions, regional patterns
2. **XXX Analysis** — computing the XXX per country, region (TBD)
3. **Hypothesis Testing** — statistical validation of the decoupling claim
4. **Visualisation** — all figures saved to `../figures/`

**Input:** `../data/clean/merged_final.csv` (produced by the merge notebook)  
**Output:** Plots saved to `../figures/`, findings summarised in final markdown cell

---
## 0. Setup & Configuration

All paths and parameters are loaded from `config.yaml` to keep the notebook reproducible and environment-independent.  
Helper functions are imported from `functions.py` to keep this notebook clean and modular.

In [3]:
# libraries
import yaml
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns


# ── Config ────────────────────────────────────────────────────────────────────
    
try:
    with open("../cfg.yaml", "r") as file:
        cfg = yaml.safe_load(file)
except:
    print("Yaml configuration file not found!")

# ── Plot style ────────────────────────────────────────────────────────────────
#sns.set_theme(style="whitegrid", palette="colorblind")
#plt.rcParams["figure.dpi"] = 120


---
## 1. Load Data

Loading the merged, cleaned dataset produced by the team merge notebook.  
We immediately validate shape, dtypes, and expected columns before proceeding.

In [4]:
#importing final dataframed cleaned and ready to work
f_df=pd.read_csv(cfg['output_data']['df3']) # ---> reading the final dataset cleaned df3 from yaml.cfg file
df=f_df.copy()
df.head(1)


,country,year,prct_access_elec,prct_renew_prod,renew_prod_kwh,prct_renew_cons,co2,co2_luc,prim_ener_cons,region,GDP,population,gdp_per_cap
0,Afghanistan,2014,89.5,2.77201,32440000.0,19.1,8.697668,8.776041,26.630217,Asia,1.950046e+10,32792523.0,594.661805


In [5]:
# ── Quick data quality check ──────────────────────────────────────────────────
print("Shape:", df.shape)
print("\nDtypes:")
print(df.dtypes)
print("\nNull counts:")
print(df.isnull().sum())

Shape: (1323, 13)

Dtypes:
country                 str
year                  int64
prct_access_elec    float64
prct_renew_prod     float64
renew_prod_kwh      float64
prct_renew_cons     float64
co2                 float64
co2_luc             float64
prim_ener_cons      float64
region                  str
GDP                 float64
population          float64
gdp_per_cap         float64
dtype: object

Null counts:
country             0
year                0
prct_access_elec    0
prct_renew_prod     0
renew_prod_kwh      0
prct_renew_cons     0
co2                 0
co2_luc             0
prim_ener_cons      0
region              0
GDP                 0
population          0
gdp_per_cap         0
dtype: int64


**Checkpoint:** Confirm the following before proceeding:
- [ ] Year range is 2014–2024
- [ ] Data types are consistent
- [ ] Key columns present: `country`, `year`, `co2`, `co2_luc`, `region`, `prim_ener_cons`,`prct_renew_prod`

---
## 2. Exploratory Data Analysis (EDA)

### 2.1 XXXX TBD

**co2_intensity**: decoupling indicator — core metric

**co2_luc_intensity**: same, honest version

**delta_co2_intensity**: rate of decoupling

**delta_renew_share**: rate of renewable adoption


In [12]:
# For the analysis we compute first the co2 intensities:
#co2 is measured in million tonnes and GDP is in international-$ in 2011 prices ($)

df['co2_int'] = df['co2']/df['GDP'] #--> scenario A Units [mill tons/$]
df['co2_luc_int'] = df['co2_luc']/df['GDP'] #--> scenario B
df.head(1)


,country,year,prct_access_elec,prct_renew_prod,renew_prod_kwh,prct_renew_cons,co2,co2_luc,prim_ener_cons,region,GDP,population,gdp_per_cap,co2_int,co2_luc_int
0,Afghanistan,2014,89.5,2.77201,32440000.0,19.1,8.697668,1.172919e-30,26.630217,Asia,1.950046e+10,32792523.0,594.661805,4.460237e-10,6.014830e-41


In [17]:
# For the deltas computation, we sort before computing 
df = df.sort_values(["country", "year"]).reset_index(drop=True)

# Delta: annual change in CO₂ intensity (Scenario A)
df["d_co2_int"] = df.groupby("country")["co2_int"].diff()  ## for the first year it give NaN

# Delta: annual change in renewable consumption share
df["d_renew_share"] = df.groupby("country")["prct_renew_cons"].diff()  ## for the first year it give NaN
df.head(5)

,country,year,prct_access_elec,prct_renew_prod,renew_prod_kwh,prct_renew_cons,co2,co2_luc,prim_ener_cons,region,GDP,population,gdp_per_cap,co2_int,co2_luc_int,d_co2_intensity,d_renew_share,d_co2_int
0,Afghanistan,2014,89.5,2.772010,32440000.0,19.1,8.697668,1.172919e-30,26.630217,Asia,1.950046e+10,32792523.0,594.661805,4.460237e-10,6.014830e-41,NaN,NaN,NaN
1,Afghanistan,2015,71.5,2.832290,33440000.0,17.7,9.384400,1.435197e-30,30.953520,Asia,1.869957e+10,33831764.0,552.722397,5.018510e-10,7.675023e-41,5.582731e-11,-1.4,5.582731e-11
2,Afghanistan,2016,97.7,3.190059,38660000.0,20.2,8.605932,1.421811e-30,28.075348,Asia,1.822435e+10,34700612.0,525.188137,4.722216e-10,7.801709e-41,-2.962940e-11,2.5,-2.962940e-11
3,Afghanistan,2017,97.7,3.168652,39470000.0,19.5,9.311054,1.350168e-30,36.517418,Asia,1.903430e+10,35688935.0,533.339054,4.891723e-10,7.093343e-41,1.695069e-11,-0.7,1.695069e-11
4,Afghanistan,2018,93.4,3.474043,38680000.0,18.3,10.191504,1.520084e-30,46.492512,Asia,1.885632e+10,36743039.0,513.194331,5.404821e-10,8.061402e-41,5.130980e-11,-1.2,5.130980e-11


### 2.2 CO₂ Trends by Region Over Time

Aggregating CO₂ per capita by region and year to spot macro-level divergence.  
Key question: Are there regions already showing a downward trend?

In [ ]:
# Aggregate: mean CO₂ per capita by region × year


### 2.3 GDP vs CO₂ Scatter — All Countries, Latest Year

A cross-sectional snapshot of where each country sits on the GDP↔CO₂ spectrum in the most recent year.  
Colour-coded by region to reveal structural patterns.

In [ ]:
# code here

---
## 3. XXX Analysis

### 3.1 Computing the XXXX

**Computing XXX** = explanation of analysis....TBD per XXXX over 2014–2024.

- **Explanation of value X** → bla bla bla → **decoupling** ✅  
- **Explanation of value X** → bla bla bla → **coupling** ❌

We define X like:  
```
x variable = asdfad - pyadlkf
```

In [ ]:
# ── Write code here ──────────────────────────────────


### 3.2 Analysis of X by X Group

Grouping countries by XXX to test whether XXXX.

In [ ]:
# ── Write code here ──────────────────────────────────


---
## 4. Hypothesis Testing

### 4.1 XXX Test Design

**XXX:** Bla bla bla
**XXX(Alternative):** Bla bla bla

**Test XXX:** Bla bla bla 
**Why:** Bla bla bla.


In [ ]:
# ── Coding, viasual etc ───────────────────────────────────────


---
## X. Summary of Findings

> **Fill this in after running all cells. Replace placeholders with your actual values.**

### Hypothesis Result

| Test | Statistic | p-value | Conclusion |
|---|---|---|---|
| Mann-Whitney U | `[U]` | `[p]` | `[Reject / Fail to reject H₀]` |
| Cohen's d | `[d]` | — | `[Small / Medium / Large]` |

### Key Observations

1. **[X group] countries** showed the highest bla bla bla `[X]%`.
2. The top blablba blab was **[country]** with a index of ... `[value]`.
3. Regional CO₂ trends show `[describe pattern]`.
4. Energy intensity `[fell / rose / was flat]` in high-income countries over 2014–2024.

### Limitations

- `prim_ener_cons` had `[X]%` nulls — imputation strategy may affect energy section results.
- Kosovo lacks an ISO code — manually assigned; verify against partner's dataset.
- 2020–2021 COVID dip artificially suppresses CO₂ — trend interpretation needs this caveat.

### Next Steps

